In [ ]:
import numpy as np
from pathlib import Path
from openmm import LangevinMiddleIntegrator, Platform, Vec3, unit
from openmm.app import CharmmParameterSet, CharmmPsfFile, HBonds, PME, Simulation

cache_dir = Path("data/gpcrmd-cache/729").resolve()
prepared_dir = Path("data/gpcrmd-mlx/729-50steps-sample50").resolve()

psf = CharmmPsfFile(str(cache_dir / "15286_dyn_729.psf"))

with np.load(prepared_dir / "prepared_system.npz", allow_pickle=False) as data:
    cell_lengths_A = np.asarray(data["cell_lengths"], dtype=float)
    positions_A = np.asarray(data["positions"], dtype=float)

psf.setBox(
    cell_lengths_A[0] * unit.angstrom,
    cell_lengths_A[1] * unit.angstrom,
    cell_lengths_A[2] * unit.angstrom,
)

parameters = CharmmParameterSet(str(cache_dir / "generated_psf_masses.rtf"))
parameters.readParameterFile(str(cache_dir / "15290_prm_729.prm"), permissive=True)

system = psf.createSystem(
    parameters,
    nonbondedMethod=PME,
    nonbondedCutoff=9.0 * unit.angstrom,
    switchDistance=7.5 * unit.angstrom,
    constraints=HBonds,
    rigidWater=True,
    ewaldErrorTolerance=5e-4,
)

dt_ps = 0.00005

integrator = LangevinMiddleIntegrator(
    310 * unit.kelvin,
    0.1 / unit.picosecond,
    dt_ps * unit.picoseconds,
)

platform = Platform.getPlatformByName("OpenCL")
simulation = Simulation(psf.topology, system, integrator, platform)

simulation.context.setPositions(
    [Vec3(*(row * 0.1)) for row in positions_A] * unit.nanometer
)

simulation.minimizeEnergy(maxIterations=2000)
simulation.context.setVelocitiesToTemperature(310 * unit.kelvin, 7)


def checked_step(simulation, steps, *, chunk=100, completed_offset=0):
    completed = 0
    while completed < steps:
        n = min(chunk, steps - completed)
        simulation.step(n)
        completed += n

        state = simulation.context.getState(getPositions=True, getEnergy=True)
        pos_A = state.getPositions(asNumpy=True).value_in_unit(unit.angstrom)
        pe = state.getPotentialEnergy().value_in_unit(unit.kilojoule_per_mole)

        if not np.isfinite(pos_A).all() or not np.isfinite(pe):
            absolute_step = completed_offset + completed
            raise RuntimeError(f"Nonfinite state after absolute step {absolute_step}")

    return completed_offset + completed


sampled_positions = []
sampled_steps = []
sampled_time_ps = []
potential_energy = []
kinetic_energy = []

total_steps = 100000
sample_count = 10
steps_per_sample = total_steps // sample_count
current_step = 0

for sample_index in range(sample_count + 1):
    state = simulation.context.getState(
        getPositions=True,
        getEnergy=True,
        enforcePeriodicBox=True,
    )

    sampled_positions.append(
        state.getPositions(asNumpy=True).value_in_unit(unit.angstrom)
    )
    sampled_steps.append(current_step)
    sampled_time_ps.append(current_step * dt_ps)
    potential_energy.append(
        state.getPotentialEnergy().value_in_unit(unit.kilojoule_per_mole)
    )
    kinetic_energy.append(
        state.getKineticEnergy().value_in_unit(unit.kilojoule_per_mole)
    )

    print(
        "sample",
        sample_index,
        "step",
        current_step,
        "PE",
        potential_energy[-1],
        "KE",
        kinetic_energy[-1],
    )

    if sample_index < sample_count:
        current_step = checked_step(
            simulation,
            steps_per_sample,
            chunk=100,
            completed_offset=current_step,
        )

sampled_positions = np.asarray(sampled_positions, dtype=np.float32)
sampled_steps = np.asarray(sampled_steps, dtype=np.int32)
sampled_time_ps = np.asarray(sampled_time_ps, dtype=np.float32)
potential_energy = np.asarray(potential_energy, dtype=np.float32)
kinetic_energy = np.asarray(kinetic_energy, dtype=np.float32)

print("sampled_positions:", sampled_positions.shape)
print("sampled_steps:", sampled_steps)
print("sampled_time_ps:", sampled_time_ps)
print("platform:", simulation.context.getPlatform().getName())


In [ ]:
with np.load(prepared_dir / "prepared_system.npz", allow_pickle=False) as data:
    symbols = np.asarray(data["symbols"]).astype(str)
    atom_names = np.asarray(data["atom_names"]).astype(str)
    residue_names = np.asarray(data["residue_names"]).astype(str)
    residue_ids = np.asarray(data["residue_ids"], dtype=np.int32)
    segment_ids = np.asarray(data["chain_ids"]).astype(str)
    ligand_indices = np.flatnonzero(np.asarray(data["ligand_mask"], dtype=bool)).astype(np.int32)
    receptor_indices = np.flatnonzero(np.asarray(data["receptor_mask"], dtype=bool)).astype(np.int32)
    water_indices = np.flatnonzero(np.asarray(data["water_mask"], dtype=bool)).astype(np.int32)
    ion_indices = np.flatnonzero(np.asarray(data["ion_mask"], dtype=bool)).astype(np.int32)
    lipid_indices = np.flatnonzero(np.asarray(data["lipid_mask"], dtype=bool)).astype(np.int32)
    cell_lengths_A = np.asarray(data["cell_lengths"], dtype=np.float32)


In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "helpers").exists():
    NOTEBOOK_DIR = Path("notebooks/ligand-receptor-motion").resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from helpers.motion_analysis import ProcessedTrajectory, ligand_com_displacement
from helpers.visualization import make_ligand_motion_figure

trajectory = ProcessedTrajectory(
    positions=sampled_positions,
    time_ps=sampled_time_ps,
    symbols=symbols,
    atom_names=atom_names,
    residue_names=residue_names,
    residue_ids=residue_ids,
    segment_ids=segment_ids,
    ligand_indices=ligand_indices,
    receptor_indices=receptor_indices,
    water_indices=water_indices,
    ion_indices=ion_indices,
    lipid_indices=lipid_indices,
    cell_lengths_A=cell_lengths_A,
    source={
        "kind": "openmm_notebook_live_sample",
        "engine": "openmm",
        "workflow": "notebook_direct_openmm_sample",
        "platform": simulation.context.getPlatform().getName(),
        "steps": int(sampled_steps[-1]),
        "frames": int(sampled_positions.shape[0]),
    },
)

disp = ligand_com_displacement(trajectory)
print("final ligand displacement A:", float(disp[-1]))
print("max ligand displacement A:", float(disp.max()))


In [ ]:
fig = make_ligand_motion_figure(
    trajectory,
    title="Live OpenMM/OpenCL 1000-step sample",
    frame_interval_ms=250,
)
fig.show()


In [ ]:
import plotly.express as px
import pandas as pd

df = pd.DataFrame({
    "step": sampled_steps,
    "time_ps": sampled_time_ps,
    "ligand_displacement_A": disp,
})

px.line(
    df,
    x="time_ps",
    y="ligand_displacement_A",
    markers=True,
    title="Ligand COM displacement over live OpenMM sample",
).show()
